In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/sk48/d8078629/sk48.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/sk48/d8078629/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/tn36/ef4dde99/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/tn36/ef4dde99/tn36.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/m0r0/492f87ba/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/m0r0/492f87ba/m0r0.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/bp35/0a0ad940/bp35.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/bp35/0a0ad940/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/cn04/2fe56bfb/cn04.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/cn04/2fe56bfb/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-

In [2]:
# ARC-AGI-3 Minimal Kaggle Notebook Template
# Simplified version for starting the competition

# =====================================================================
# STEP 1: Install dependencies from wheels (no internet needed)
# =====================================================================
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

Looking in links: /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arc_agi-0.9.8-py3-none-any.whl
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arcengine-0.9.3-py3-none-any.whl (from arc-agi)
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/pillow-12.2.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (from arc-agi)
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 

In [3]:
import os
import shutil

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    src = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents'
    dst = '/kaggle/working/ARC-AGI-3-Agents'
    
    if not os.path.exists(dst):
        print(f"Copying {src} to {dst}")
        shutil.copytree(src, dst)
    else:
        print(f"Already exists: {dst}")
    
    print("Waiting for gateway...")
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games
else:
    # Local mode: create the directory manually
    dst = '/kaggle/working/ARC-AGI-3-Agents'
    if not os.path.exists(dst):
        print(f"Creating {dst} (local mode)")
        os.makedirs(dst, exist_ok=True)
        # Also create the templates folder
        os.makedirs(f'{dst}/agents/templates', exist_ok=True)
        print("Created directories for local mode")
    else:
        print(f"Directory already exists: {dst}")

Creating /kaggle/working/ARC-AGI-3-Agents (local mode)
Created directories for local mode


In [4]:
%%writefile /kaggle/working/ARC-AGI-3-Agents/agents/templates/simple_agent.py
from typing import Any
import random
import numpy as np

from agents.agent import Agent
from arcengine import GameAction, GameState


class SimpleAgent(Agent):
    MAX_ACTIONS_PER_LEVEL = 200
    
    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self.action_counter = 0
        self.actions_tried = set()
        
    def is_done(self, frames, latest_frame) -> bool:
        if latest_frame.state == GameState.WIN:
            return True
        if self.action_counter > self.MAX_ACTIONS_PER_LEVEL:
            return True
        if latest_frame.state == GameState.GAME_OVER:
            return True
        return False
    
    def choose_action(self, frames, latest_frame) -> GameAction:
        if latest_frame.state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
            self.action_counter = 0
            self.actions_tried.clear()
            action = GameAction.RESET
            action.reasoning = "Reset game to start."
            return action
        
        available = getattr(latest_frame, 'available_actions', None)
        if available:
            action_ids = [a.value if hasattr(a, 'value') else int(a) for a in available]
        else:
            action_ids = [1, 2, 3, 4, 5, 6]
        
        for action_id in action_ids:
            if action_id not in self.actions_tried and self.action_counter < 50:
                self.actions_tried.add(action_id)
                
                if action_id == 6:
                    action = GameAction.ACTION6
                    action.set_data({"x": random.randint(0, 10), "y": random.randint(0, 10)})
                    action.reasoning = f"Probe ACTION6 at (x,y)"
                else:
                    action = GameAction(action_id)
                    action.reasoning = f"Probe ACTION{action_id}"
                
                self.action_counter += 1
                return action
        
        if self.action_counter < self.MAX_ACTIONS_PER_LEVEL:
            action_id = (self.action_counter % 5) + 1
            action = GameAction(action_id)
            action.reasoning = f"Heuristic: ACTION{action_id}"
            self.action_counter += 1
            return action
        
        action = GameAction.RESET
        action.reasoning = "Too many actions, reset."
        return action

Writing /kaggle/working/ARC-AGI-3-Agents/agents/templates/simple_agent.py


In [5]:
# =====================================================================
# STEP 4: Simplify __init__.py to avoid heavy imports
# =====================================================================
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    init_code = '''from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.simple_agent import SimpleAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "simple": SimpleAgent,
}
'''
    
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write(init_code)
    
    env_code = '''SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
'''
    
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write(env_code)

In [6]:

# =====================================================================
# STEP 5: Run agent (competition) or create dummy submission (local)
# =====================================================================
import pandas as pd

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print("Running SimpleAgent on competition games...")
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent simple
else:
    print("Creating dummy submission for local testing...")
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    print(submission.head())

Creating dummy submission for local testing...
  row_id game_id  end_of_game  score
0    1_0       1         True      1
